In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import io

paper_data_text = """Methods,Confidence_MNIST,Confidence_CIFAR10,AUROC_MNIST,AUROC_CIFAR10
MAP,75.0±0.4,76.1±1.2,96.5±0.1,92.1±0.5
DE,65.7±0.3,65.4±0.4,97.5±0.0,94.0±0.1
VB,73.2±0.8,58.8±0.7,95.8±0.2,88.7±0.3
HMC,69.2±1.7,69.4±0.6,96.1±0.2,90.6±0.2
SWG,75.8±0.3,68.1±2.3,96.5±0.1,91.3±0.8
LA,67.5±0.4,69.0±1.3,96.2±0.2,92.2±0.5
LA_STAR,56.1±0.5,55.7±1.2,96.4±0.2,92.4±0.5
"""

df_paper = pd.read_csv(io.StringIO(paper_data_text))

def split_mean_error(df, columns):
    for col in columns:
        df[[col, f'{col}_err']] = df[col].str.split('±', expand=True)
        df[col] = pd.to_numeric(df[col])
        df[f'{col}_err'] = pd.to_numeric(df[f'{col}_err'])
    return df

df_paper = split_mean_error(df_paper, ['Confidence_MNIST', 'Confidence_CIFAR10', 'AUROC_MNIST', 'AUROC_CIFAR10'])

df_paper_mnist = df_paper[['Methods', 'Confidence_MNIST', 'Confidence_MNIST_err', 'AUROC_MNIST', 'AUROC_MNIST_err']].copy()
df_paper_cifar10 = df_paper[['Methods', 'Confidence_CIFAR10', 'Confidence_CIFAR10_err', 'AUROC_CIFAR10', 'AUROC_CIFAR10_err']].copy()

df_paper_mnist.columns = ['Methods', 'Confidence', 'Confidence_err', 'AUROC', 'AUROC_err']
df_paper_cifar10.columns = ['Methods', 'Confidence', 'Confidence_err', 'AUROC', 'AUROC_err']

def process_user_data(filepath):
    df = pd.read_csv(filepath, header=None)
    
    part1 = df[[0, 3, 4]].copy()
    part1.columns = ['Methods', 'Confidence', 'AUROC']
    
    part2 = df[[6, 8, 9]].copy()
    part2.columns = ['Methods', 'Confidence', 'AUROC']
    
    part3 = df[[11, 12, 13]].copy()
    part3.columns = ['Methods', 'Confidence', 'AUROC']
    
    combined_df = pd.concat([part1, part2, part3]).dropna()
    
    agg_df = combined_df.groupby('Methods').agg({ 'Confidence': 'mean', 'AUROC': 'mean' }).reset_index()
    
    return agg_df

df_user_mnist = process_user_data('table1_MNIST.csv')
df_user_cifar10 = process_user_data('table1_CIFAR10.csv')

df_paper_mnist['Source'] = 'Paper'
df_paper_cifar10['Source'] = 'Paper'
df_user_mnist['Source'] = 'Ours'
df_user_cifar10['Source'] = 'Ours'

df_mnist = pd.concat([df_paper_mnist, df_user_mnist], ignore_index=True)
df_cifar10 = pd.concat([df_paper_cifar10, df_user_cifar10], ignore_index=True)

df_mnist['Methods'] = df_mnist['Methods'].str.upper()
df_cifar10['Methods'] = df_cifar10['Methods'].str.upper()
df_mnist['Methods'] = df_mnist['Methods'].replace({'LA*':'LA*'})
df_cifar10['Methods'] = df_cifar10['Methods'].replace({'LA*':'LA*'})

def plot_comparison(df, dataset_name):
    methods = sorted(list(df['Methods'].unique()))
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f'{dataset_name} OOD Detection Performance Comparison', fontsize=16)

    bar_width = 0.35
    index = np.arange(len(methods))

    confidence_pivot = df.pivot_table(index='Methods', columns='Source', values='Confidence').reindex(methods)
    confidence_err_paper = df[df['Source'] == 'Paper'].set_index('Methods').reindex(methods)['Confidence_err']
    
    axes[0].bar(index - bar_width/2, confidence_pivot['Paper'], bar_width, yerr=confidence_err_paper, capsize=5, label='Paper')
    axes[0].bar(index + bar_width/2, confidence_pivot['Ours'], bar_width, label='Ours (Reproduced)')

    axes[0].set_ylabel('Confidence (↓)')
    axes[0].set_title('Confidence Comparison')
    axes[0].set_xticks(index)
    axes[0].set_xticklabels(methods, rotation=45, ha="right")
    axes[0].legend()
    
    auroc_pivot = df.pivot_table(index='Methods', columns='Source', values='AUROC').reindex(methods)
    auroc_err_paper = df[df['Source'] == 'Paper'].set_index('Methods').reindex(methods)['AUROC_err']

    axes[1].bar(index - bar_width/2, auroc_pivot['Paper'], bar_width, yerr=auroc_err_paper, capsize=5, label='Paper')
    axes[1].bar(index + bar_width/2, auroc_pivot['Ours'], bar_width, label='Ours (Reproduced)')

    axes[1].set_ylabel('AUROC (%) (↑)')
    axes[1].set_title('AUROC Comparison')
    axes[1].set_xticks(index)
    axes[1].set_xticklabels(methods, rotation=45, ha="right")
    axes[1].legend()
    if dataset_name.lower().startswith('cifar'):
        axes[1].set_ylim(bottom=60)
    else:
        axes[1].set_ylim(bottom=85)

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'{dataset_name}_comparison.png')
    plt.close()

plot_comparison(df_mnist, 'MNIST')
plot_comparison(df_cifar10, 'CIFAR-10')